# ဂိုတယ်မှာ အခန်းမှာယူခြင်းနှင့် ဦးစားပေးအဖွဲ့ဝင် Middleware

ဤ notebook သည် Microsoft Agent Framework ကို အသုံးပြု၍ **function-based middleware** ကို ဖော်ပြပါသည်။ ဤသည်သည် အခြေအနေအလိုက် အလုပ်စဉ် ဥပမာတွင် middleware အလွှာတစ်ခု ထပ်ထည့်ခြင်းဖြင့် ဦးစားပေး အဖွဲ့ဝင်များအတွက် အထူးအခွင့်အရေးများ ပေးသည်။

## သင်လေ့လာရန်:
1. **Function-Based Middleware**: Function ရလဒ်များကို တွန်းလှန်တားဆီးပြီး ပြင်ဆင်ခြင်း
2. **Context Access**: အလုပ်ဆောင်ပြီးနောက် `context.result` ကို ဖတ်ခြင်းနှင့် ပြင်ဆင်ခြင်း
3. **စီးပွားရေးအတွင်းရေးဆောင်ရွက်ခြင်း**: ဦးစားပေး အဖွဲ့ဝင် အကျိုးခံစားခွင့်များ
4. **ရလဒ် ပယ်ချခြင်း**: အသုံးပြုသူ အခြေအနေ အရ function ရလဒ် ပြောင်းလဲခြင်း
5. **အလုပ်စဉ်တူညီခြင်း၊ ရလဒ်မတူခြင်း**: Middleware မှ တာဝန် ထမ်းဆောင်မှုမှ ဖြစ်ပေါ်သော ပြင်ဆင်မှုများ

## Middleware ပါသော Workflow မျှော်စင်ကွဲ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## အခြေအနေအလိုက် အလုပ်စဉ်နှင့် ကွဲပြားချက် အဓိက:

**Middleware မပါလျှင်** (14-conditional-workflow.ipynb):
- ပြဲရီဆ်တွင် အခန်းမရှိ → alternative_agent သို့ သွားခြင်း

**Middleware ပါလျှင်** (ဤ notebook):
- ပုံမှန်အသုံးပြုသူ + ပြဲရီဆ် → အခန်း မရှိ → alternative_agent သို့ သွားခြင်း
- ဦးစားပေးအသုံးပြုသူ + ပြဲရီဆ် → 🌟 Middleware က override လုပ်သည်! → အခန်း ရနိုင် → booking_agent သို့ သွားခြင်း

## လိုအပ်ချက်များ:
- Microsoft Agent Framework ထည့်သွင်းထားခြင်း
- အခြေအနေအလိုက် အလုပ်စဉ် များနားလည်မှု (14-conditional-workflow.ipynb ကို ကြည့်ရန်)
- GitHub token သို့မဟုတ် OpenAI API key
- Middleware ပုံစံများ အခြေခံနားလည်မှု


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## အဆင့် ၁: ဖွဲ့စည်းထားသော ရလဒ်များအတွက် Pydantic မော်ဒယ်များ သတ်မှတ်ခြင်း

ဤမော်ဒယ်များသည် ကိုယ်စားလှယ်များ ပြန်လည်ပေးပို့မည့် **schema** ကို သတ်မှတ်သည်။ မိုင်းခွိတ်ဝဲသည် ရရှိနိုင်မှုရလဒ်ကို ပြင်ဆင်သောအခါ ထိန်းသိမ်းရန် `priority_override` ဖြည့်စွက်ထားသည်။


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: ဦးစားပေး सदस्यများ ဒေတာဘေ့စ်ကို သတ်မှတ်ပါ

ဒီဒိုမိုးအတွက် ဦးစားပေး အဖွဲ့ဝင်ဒေတာဘေ့စ်ကို စိန့်ဂျူလေးရှင်းလုပ်မည်ဖြစ်သည်။ ထုတ်ကုန်ထိန်းသိမ်းမှုတွင်၊ ဤသည်သည် အမှန်တကယ်သော ဒေတာဘေ့စ် သို့မဟုတ် API ကို မေးမြန်းမည်ဖြစ်သည်။

**ဦးစားပေး सदस्यများ:**
- `alice@example.com` - VIP အဖွဲ့ဝင်
- `bob@example.com` - Premium အဖွဲ့ဝင်  
- `priority_user` - စမ်းသပ်ရေးအကောင့်


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## အဆင့် ၃: ဟိုတယ်ဘွတ်ကင်ကိရိယာ ဖန်တီးခြင်း

အသတ်မှတ်လုပ်ဆောင်ချက်တွေနဲ့တူပေမယ့် လောလောဆယ် middleware က မှတ်တားမယ်!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## အဆင့် ၄: 🌟 ဦးစားပေး စစ်ဆေးမှု Middleware ကို ဖန်တီးပါ (အဓိက အစိတ်အပိုင်း!)

ဒီ notebook ရဲ့ **အဓိက လုပ်ဆောင်ချက်** ဖြစ်ပါတယ်။ Middleware က:

1. hotel_booking အလုပ်လုပ်မှု ကို **ကြားနာ** လုပ်ဆောင်သည်
2. `next(context)` ကို ခေါ်ပြီး လုပ်ဆောင်မှုကို စနစ်တကျ **စီမံ** လုပ်ဆောင်သည်
3. `context.result` ထဲက ရလဒ်ကို **စစ်ဆေး** လုပ်ဆောင်သည်
4. အသုံးပြုသူသည် ဦးစားပေး ဖြစ်ပြီး အခန်း မရှိပါက ရလဒ်ကို **အစားထိုး** ပေးသည်
5. ပြင်ဆင်ပြီးသော ရလဒ်ကို ပြန်လည် **ပေးပို့** လုပ်ဆောင်သည်

**အဓိက ပုံစံ:**
```python
async def my_middleware(context, next):
    await next(context)  # ဖန်ဆင်းမှုကို အကောင်အထည်ဖော်ပါ
    # ယခု context.result ထဲတွင် ဖန်ဆင်းမှု၏ အထွက်ရှိသည်
    if some_condition:
        context.result = new_value  # အစားထိုးပါ!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## အဆင့် ၅: လမ်းညွှန်ခြင်းအတွက် အခြေအနေ ဖန်ရှင်များ သတ်မှတ်ခြင်း

conditional workflow နှင့်တူညီသော အခြေအနေ ဖန်ရှင်များဖြစ်ပြီး၊ ၎င်းတို့သည် လမ်းညွှန်မှုကို ဆုံးဖြတ်ရန် အဖွဲ့စည်းတကျထွက်ရှိမှုကို စစ်ဆေးကြသည်။


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## အဆင့် 6: အသုံးပြုသူပြသမှု Executor ကို ဖန်တီးပါ

မပြီးခဲ့သည့် executor နှင့် တူညီသည် - နောက်ဆုံး workflow အထွက်ကို ပြသသည်။


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## အဆင့် ၇: ပတ်ဝန်းကျင်အပြောင်းအလဲများအား ဖတ်ယူပါ

LLM client ကို ချိန်ညှိပါ (GitHub Models သို့မဟုတ် OpenAI)။


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## အဆင့် ၈: Middleware ဖြင့် AI Agent များ ဖန်တီးခြင်း

**အဓိက ကွာခြားချက်:** availability_agent ကို ဖန်တီးသည့်အခါ `middleware` ပေးပို့ပါသည်!

ဤကဲ့သို့ priority_check_middleware ကို agent ၏ function ဖုန်းခေါ်မှု လမ်းကြောင်းထဲသို့ ထည့်သွင်းသည်။


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: Workflow ကို တည်ဆောက်ခြင်း

ယခင်အတိုင်း workflow ဖွဲ့စည်းပုံတူသည် - ရရှိနိုင်မှုအပေါ် မှီငြမ်းသော လမ်းညွှန်မှု။


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## အဆင့် 10: စမ်းသပ်မှု ကိစ္စ 1 - ပဲရစ်ရှိ သာမန်အသုံးပြုသူ (အစားထိုးမှုမရှိ)

သာမန်အသုံးပြုသူတစ်ဦးသည် ပဲရစ်ကို စီစဉ်ကြိုးစားသည် → အခန်းမရှိခြင်း → alternative_agent သို့ လမ်းကြောင်းပြောင်းရွှေ့သည်


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 Priority User in Paris (WITH Override!)

A priority member tries to book Paris → No rooms initially → 🌟 Middleware overrides! → Routes to booking_agent

**ဤသည်မှာ middleware အင်အား၏ အဓိကပြသချက်ဖြစ်သည်!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## အဆင့် 12: စမ်းသပ်မှု ကိစ္စ 3 - စတော့့က်ဟုံးရှိ ဦးစားပေးအသုံးပြုသူ (ပြီးသားရှိပြီး)

ဦးစားပေး အသုံးပြုသူသည် စတော့့က်ဟုံးကို ကြိုးစားသည် → အခန်းများ ရနိုင်သည် → မလိုအပ်သော override မရှိပါ → booking_agent သို့ မောင်းနှင်သည်

ဒီသည် middleware သည် လိုအပ်သောအခါတွင်သာ လှုပ်ရှားသည်ကို ပြသည်!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## အဓိကယူဆချက်များနှင့် Middleware အယူအဆများ

### ✅ သင်ယူသင့်သည်များ

#### **1. Function-Based Middleware ပုံစံ**

Middleware သည် function ခေါ်ဆိုမှုများကို ရိုးရှင်းသော async function ဖြင့် ရပ်တန့်ဆောင်ရွက်သည်။

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ဖန်ရှင်ကို ဆောင်ရွက်မည့်အချိန်မတိုင်မှီ
    print("Intercepting...")
    
    # ဖန်ရှင်ကို ဆောင်ရွက်ပါ
    await next(context)
    
    # ဖန်ရှင်ဆောင်ရွက်ပြီးပါက ရလဒ်ကို စစ်ဆေးပါ
    if context.result:
        # လိုအပ်လျှင် ရလဒ်ကို ပြုပြင်ပါ
        context.result = modified_value
```

#### **2. အကြောင်းအရာ ရယူခြင်းနှင့် ရလဒ် ပြောင်းလဲရေး**

- `context.function` - ခေါ်ဆိုနေသော function ကို ရယူခြင်း
- `context.arguments` - function ၏ arguments များကို ဖတ်ခြင်း
- `context.kwargs` - ပေါင်းထည့်သတ်မှတ်ချက်များကို ရယူခြင်း
- `await next(context)` - function ကို အကောင်အထည်ဖော်ခြင်း
- `context.result` - function ၏ output ကို ဖတ်/ပြောင်းလဲခြင်း

#### **3. စီးပွားရေးထုတ်လုပ်မှု လုပ်ဆောင်ချက်**

ကျွန်ုပ်တို့၏ middleware သည် ဦးစားပေးအဖွဲ့ဝင် အကျိုးခံစားခွင့်များကို အကောင်အထည်ဖော်သည်-
- **ပုံမှန်အသုံးပြုသူများ**: မပြောင်းလဲခြင်း၊ စံနှုန်း workflow
- **ဦးစားပေးအသုံးပြုသူများ**: "မရှိပါ" ကို "ရှိသည်" အဖြစ် ပြောင်းလဲခြင်း
- **အခြေအနေ logic**: လိုအပ်သည့်အချိန်တွင်သာ ပြောင်းလဲပေးသည်

#### **4. Workflow တူညီသော်လည်း အကျိုးသက်ရောက်မှု မတူကြောင်း**

Middleware ၏ အင်အား-
- ✅ Workflow ဖွဲ့စည်းမှု မပြောင်းလဲပါ
- ✅ Tool function မပြောင်းလဲပါ
- ✅ အခြေအနေဆိုင်ရာ routing logic မပြောင်းလဲပါ
- ✅ Middleware သာဖြင့် → လုပ်ဆောင်ချက် မတူကြောင်း!

### 🚀 အမှန်တကယ် အသုံးများမှုများ

1. **VIP/Premium အဖွဲ့အစည်းများအတွက်**
   - Premium အသုံးပြုသူများအတွက် အမြန်နှုန်းကန့်သတ်ချက် ပြောင်းလဲရန်
   - အရင်းအနှီးသို့ ဦးစားပေး ရောက်ရှိခွင့်ပေးရန်
   - Premium လုပ်ဆောင်ချက်များကို ယုံကြည်စိတ်ချစွာ ဖြည့်ဆည်းခြင်း

2. **A/B စမ်းသပ်မှုများ**
   - အသုံးပြုသူများအား အမျိုးမျိုးသော အကောင်အထည်ဖော်မှုများသို့ လမ်းညွှန်ခြင်း
   - အသီးသီးသော အသုံးပြုသူများနှင့် နည်းပညာ အသစ်များ စမ်းသပ်ခြင်း
   - လျော့လျော့ နည်းနည်း အဆင့်ဆင့် လုပ်ဆောင်ခြင်း

3. **လုံခြုံရေးနှင့် စည်းမျဉ်းစည်းကမ်း**
   - Function ခေါ်ဆိုမှုများ စစ်ဆေးခြင်း
   - အရေးကြီး လုပ်ငန်းစဉ်များ ပိတ်ဆို့ခြင်း
   - စီးပွားရေးစည်းမျဉ်းများ အကောင်အထည်ဖော်ခြင်း

4. **လုပ်ဆောင်မှု မြှင့်တင်ခြင်း**
   - သတ်မှတ်သော အသုံးပြုသူများအတွက် ရလဒ်များ Cache ထားခြင်း
   - အလေးချိန်ကြီးသော လုပ်ဆောင်ချက်များကို မလိုအပ်ပါက ကျော်သွားခြင်း
   - အရင်းအနှီး dynamic လစာခြင်း

5. **အမှား စီမံခန့်ခွဲခြင်းနှင့် ပြန်လည်ကြိုးစားခြင်း**
   - အမှားများကို ဖမ်းထားပြီး ကျင့်သုံးခြင်း
   - ပြန်လည်ကြိုးစားမှု Logic အကောင်အထည်ဖော်ခြင်း
   - အခြားနည်းလမ်းများကို အစားထိုးအသုံးပြုခြင်း

6. **မှတ်တမ်းတင်ခြင်းနှင့် စစ်ဆေးခြင်း**
   - Function လုပ်ဆောင်ချိန်များကို စောင့်ကြပ်ခြင်း
   - Parameters နှင့် ရလဒ်များ မှတ်တမ်းတင်ခြင်း
   - အသုံးပြုမှု ပုံစံများ စောင့်ကြပ်ခြင်း

### 🔑 Decorator မတူသော အချက်အလက်များ

| လက္ခဏာ | Decorator | Middleware |
|---------|-----------|------------|
| **အနှံ့အယှက်** | တစ်ခုတည်းသော function | Agent တွင် function အားလုံး |
| **လွတ်လပ်ခွင့်** | သတ်မှတ်ချက်တွင် ကန့်သတ် | Runtime တွင် dynamic |
| **အကြောင်းအရာ** | ကန့်သတ်ထားသည် | Agent အပြည့်အစုံ Context |
| **ပေါင်းစပ်မှု** | Decorator များစွာ | Middleware Pipeline |
| **Agent အသိအမှတ်ပြုမှု** | မရှိ | ရှိ (Agent အခြေအနေ ဂရုစိုက်) |

### 📚 Middleware အသုံးပြုရန် အချိန်

✅ **Middleware ကို အသုံးပြုပါက:**
- အသုံးပြုသူ/Session အခြေအနေ အပေါ်မူတည်ပြီး လုပ်ဆောင်ချက် ပြောင်းလဲလိုသည်
- Function များစွာတွင် Logic အသုံးပြုလိုသည်
- Agent အဆင့်အခြေအနေကို ရယူရန်လိုသည်
- လက်တွေ့စပ်လျဉ်း ဆောင်ရွက်မှုများ (Logging, Auth, စသည်) ကို အကောင်အထည်ဖော်ရန်

❌ **Middleware မသုံးသင့်သည့်အချိန်**
- ရိုးရှင်းသော input စစ်ဆေးခြင်း (Pydantic အသုံးပြုပါ)
- Function သီးသန့် Logic (Function တွင် ထားသင့်သည်)
- တစ်ကြိမ်ပြောင်းလဲမှုများ (သာ Function ကို ပြောင်းလဲပါ)

### 🎓 ဦးဆောင်ပုံစံများ

```python
# မတည့်သော middleware များ (အကောင်အထည်ဖော်သည့်အဆင့်စဉ် အရေးကြီးသည်!)
middleware=[
    logging_middleware,      # အရင်ဆုံး မှတ်တမ်းတင်သည်
    auth_middleware,         # ထို့နောက် အတည်ပြုချက်စစ်ဆေးသည်
    cache_middleware,        # ထို့နောက် cache ကို စစ်ဆေးသည်
    rate_limit_middleware,   # ထို့နောက် အတိုင်းအတာကန့်သတ်ချက်
    priority_check_middleware  # နောက်ဆုံးတွင် ဦးစားပေးစစ်ဆေးမှု
]

# အခြေအနေအရ middleware အကောင်အထည်ဖော်မှု
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ရလဒ်ကို ပြင်ဆင်ပါ
    else:
        # အကောင်အထည်ဖော်မှုကို အတိအကျ မကျော်ဖြတ်ပါနဲ့
        context.result = cached_value
```

### 🔗 ဆက်စပ်အယူအဆများ

- **Agent Middleware**: agent.run() ခေါ်ဆိုမှုများကို ရပ်တန့်ခြင်း
- **Function Middleware**: tool function ခေါ်ဆိုမှုများကို ရပ်တန့်ခြင်း (ကျွန်ုပ်တို့အသုံးပြုပုံ!)
- **Middleware Pipeline**: အဆင့်လိုက် Middleware များဆက်တိုက် လုပ်ဆောင်ခြင်း
- **Context Propagation**: Middleware ချည်ဆက်တန်းမှတဆင့် အခြေအနေ ပေးပို့ခြင်း


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
